# PACE — Phase 11E2 Git LFS fix / direct AMPGAN v2 inference

**Analysis only:** original manuscript, supplementary data and reviewer response files remain untouched.

Your prior Phase 11E2 results confirmed source commit `2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70`, but stopped at `ModuleNotFoundError`-equivalent `git: lfs is not a git command`. This notebook installs Git LFS automatically if missing and retries the official checkpoint + direct TensorFlow SavedModel inference. A completed model-generation run is NOT guaranteed: the original model may have additional TensorFlow compatibility or conditioning-data dependencies. Those are reported transparently.

**Instructions:** open in Google Colab, click **Runtime → Run all**, download `PACE_Phase11E2_Results.zip` and upload it in the conversation. No PACE dataset ZIPs required. If same runtime is active, it reuses the already-cloned official repository and any downloaded checkpoint weights.


## 1. Install lightweight dependencies and verify Git LFS

The Git LFS install is also automatically checked inside the runner below, so restarting or skipping this cell does not silently bypass that dependency.


In [ ]:
%pip -q install biopython
import subprocess
check = subprocess.run(['git','lfs','version'],capture_output=True,text=True)
if check.returncode:
    print('Git LFS not installed. Installing now...')
    subprocess.run(['apt-get','update','-qq'],check=False)
    subprocess.run(['apt-get','install','-y','-qq','git-lfs'],check=True)
subprocess.run(['git','lfs','version'],check=True)
print('Dependency check finished. Do not restart runtime.')


## 2. Official-source checkpoint verification and direct SavedModel probe

The source commit and all four checkpoint hashes are verified before any inference. The official, training-derived conditioning dataset is used if it is available. No fabricated peptide sequences, conditioning labels or performance claims are produced. If inference cannot proceed, this step still writes a diagnostic result ZIP.


In [ ]:
from pathlib import Path
RUNNER_SOURCE = '"""PACE Phase 11E2: source-authenticated AMPGANv2 recovery and inference.\n\nAnalysis only. Never edit PACE manuscript, supplements or reviewer responses.\nUses exact upstream GitLab v2.0 weights, and (if available) conditioning\nvectors obtained from upstream source data; no fabricated positive candidates.\nModel weights and raw training sources are excluded from the results archive.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, importlib, json, os, platform, re, subprocess, sys, time, traceback, zipfile\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nROOT=Path(\'/content\') if Path(\'/content\').is_dir() else Path.cwd()\nREPO=ROOT/\'ampgan_v2_official\'\nOUT=ROOT/\'PACE_Phase11E2_Results\'\nOUT.mkdir(parents=True,exist_ok=True)\nZIP=ROOT/\'PACE_Phase11E2_Results.zip\'\nOFFICIAL=\'https://gitlab.com/vail-uvm/amp-gan.git\'\nCOMMIT=\'2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\'\nTAG=\'v2.0\'\nMODEL_REL=\'models/amp_gan_2020-07-08_24/gan_1606\'\nEXPECTED={\n MODEL_REL+\'/saved_model.pb\':(\'a5e7cafa16c33010bd0fe22747c89f05d74f8781deac9eeeb91c6e4c371177e2\',1344320),\n MODEL_REL+\'/variables/variables.data-00000-of-00002\':(\'df6b16408b05f21a6fdb1d675c12562257d619382d08af2fad391d7ed847084a\',53347),\n MODEL_REL+\'/variables/variables.data-00001-of-00002\':(\'012fc17624f1fe6b0441622bebc71ee968e4d7db80bb7910e22af93b49d69890\',12982248),\n MODEL_REL+\'/variables/variables.index\':(\'7936298d0a7ac1802bbedc726d29e27911c3aa60778ac97395a24d5f82456902\',12646)\n}\nN_GENERATE=int(os.environ.get(\'PACE_AMPGAN_N\',\'1000\'))\nBATCH_SIZE=int(os.environ.get(\'PACE_AMPGAN_BATCH\',\'128\'))\nDATA_CAP_BYTES=int(os.environ.get(\'PACE_AMPGAN_DATA_CAP_MB\',\'700\'))*1024*1024\nSEED=20261114\nreport={\'phase\':\'11E2\',\'status\':\'initializing\',\'timestamp_utc\':datetime.now(timezone.utc).isoformat(),\n        \'source\':OFFICIAL,\'tag\':TAG,\'expected_commit\':COMMIT,\'python\':sys.version,\n        \'platform\':platform.platform(),\'n_requested\':N_GENERATE,\'seed\':SEED,\n        \'generated_count\':0,\'checkpoint\':{},\'data_inventory\':{},\'steps\':{},\n        \'notes\':[\'Original PACE input files untouched. Do not treat model predictions as experimental MIC or toxicity.\']}\nLOG=[]\n\ndef write_log(t):\n    msg=str(t);LOG.append(msg);print(msg,flush=True)\n\ndef cmd(args,cwd=None,timeout=3600,env=None,check=True):\n    write_log(\'RUN: \'+\' \'.join(map(str,args)))\n    p=subprocess.run(args,cwd=str(cwd) if cwd else None,\n                     env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)\n    write_log(\'exit=%s\\n%s\'%(p.returncode,(p.stdout or \'\')[-5000:]))\n    if check and p.returncode: raise RuntimeError(\'Command failed: \'+str(args))\n    return p\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef ensure_git_lfs():\n    # Colab runtimes do not always provide Git LFS by default.\n    pre=cmd([\'git\',\'lfs\',\'version\'],check=False)\n    if pre.returncode:\n        write_log(\'Git LFS missing; installing official Ubuntu git-lfs package...\')\n        # Colab third-party apt sources may print benign warnings.\n        cmd([\'apt-get\',\'update\',\'-qq\'],timeout=600,check=False)\n        cmd([\'apt-get\',\'install\',\'-y\',\'-qq\',\'git-lfs\'],timeout=900,check=True)\n        cmd([\'git\',\'lfs\',\'version\'],check=True)\n    report[\'steps\'][\'git_lfs\']=\'ready\'\n\ndef verify_checkpoint():\n    ensure_git_lfs()\n    if not REPO.is_dir():\n        e=os.environ.copy();e[\'GIT_LFS_SKIP_SMUDGE\']=\'1\'\n        cmd([\'git\',\'clone\',\'--branch\',TAG,\'--depth\',\'1\',OFFICIAL,str(REPO)],env=e)\n    git_commit=cmd([\'git\',\'rev-parse\',\'HEAD\'],cwd=REPO).stdout.strip()\n    report[\'actual_commit\']=git_commit\n    if git_commit!=COMMIT: raise RuntimeError(\'Unexpected repository commit: \'+git_commit)\n    cmd([\'git\',\'lfs\',\'install\',\'--local\'],cwd=REPO,check=True)\n    missing=[]\n    for rel,(hash_expected,byt_expected) in EXPECTED.items():\n        p=REPO/rel\n        if not p.exists() or p.stat().st_size!=byt_expected or sha(p)!=hash_expected:missing.append(rel)\n    if missing:\n        cmd([\'git\',\'lfs\',\'pull\',\'--include\',MODEL_REL+\'/**\',\'--exclude\',\'\'],cwd=REPO,timeout=2400)\n    for rel,(hash_expected,byt_expected) in EXPECTED.items():\n        p=REPO/rel\n        info={\'exists\':p.exists(),\'size\':p.stat().st_size if p.exists() else None,\n              \'sha256\':sha(p) if p.exists() else None}\n        info[\'verified\']=info[\'size\']==byt_expected and info[\'sha256\']==hash_expected\n        report[\'checkpoint\'][rel]=info\n        if not info[\'verified\']:raise RuntimeError(\'Checkpoint mismatch: \'+rel)\n    write_log(\'All four checkpoint files independently SHA256 verified.\')\n    report[\'steps\'][\'checkpoint\']=\'passed\'\n\ndef save_savedmodel_inspection(root):\n    inspect={\'object_type\':str(type(root)),\n             \'public_attributes\':[a for a in dir(root) if not a.startswith(\'_\')][:100]}\n    if hasattr(root,\'signatures\'):\n        inspect[\'signatures\']={k:str(v.structured_input_signature) for k,v in root.signatures.items()}\n    if hasattr(root,\'generator\'):\n        g=root.generator\n        inspect[\'generator_type\']=str(type(g))\n        inspect[\'generator_callable\']=callable(g)\n        if hasattr(g,\'signatures\'):\n            inspect[\'generator_signatures\']={k:str(v.structured_input_signature) for k,v in g.signatures.items()}\n    (OUT/\'savedmodel_inspection.json\').write_text(json.dumps(inspect,indent=2,default=str))\n    return inspect\n\ndef get_model():\n    import tensorflow as tf\n    report[\'tensorflow_version\']=tf.__version__\n    write_log(\'Direct TensorFlow SavedModel load (no vailtools/model.py imported)...\')\n    m=tf.saved_model.load(str(REPO/MODEL_REL))\n    inspect=save_savedmodel_inspection(m)\n    if not hasattr(m,\'generator\') or not callable(m.generator):\n        report[\'steps\'][\'model\']=\'no_callable_generator\';raise RuntimeError(\'Restored SavedModel has no callable generator; inspect signatures\')\n    report[\'steps\'][\'model\']=\'passed\'\n    return tf,m.generator\n\ndef get_source_conditions():\n    """Only original upstream training labels; never invent conditioning vectors."""\n    # Detect LFS dataset members from exact tagged source. Restrict downloads to\n    # official source data modules and cap total bytes to prevent surprises.\n    source_dir=REPO/\'ampgan\'\n    source_parts=[\'dbaasp.py\',\'avpdb.py\',\'uniprot.py\',\'data_utils.py\']\n    source_text=\'\\n\'.join((source_dir/p).read_text(errors=\'replace\') for p in source_parts)\n    dirs=set(re.findall(r\'\\.\\./data/([a-zA-Z0-9_-]+)/\',source_text))\n    dirs.update([\'dbaasp\',\'avpdb\',\'uniprot\'])\n    filelist=cmd([\'git\',\'ls-files\'],cwd=REPO).stdout.splitlines()\n    lfs=[]\n    for rel in filelist:\n        if not rel.startswith(\'data/\') or len(rel.split(\'/\'))<3:continue\n        d=rel.split(\'/\')[1]\n        if d.lower() not in {x.lower() for x in dirs}:continue\n        path=REPO/rel\n        if not path.exists():continue\n        with open(path,\'rb\') as f:probe=f.read(200).decode(errors=\'ignore\')\n        if probe.startswith(\'version https://git-lfs.github.com/spec/v1\'):\n            z=re.search(r\'(?m)^size (\\d+)\',probe)\n            if z:lfs.append((rel,int(z.group(1))))\n    total=sum(size for _,size in lfs)\n    report[\'data_inventory\']={\'official_data_directories\':sorted(dirs),\n                              \'lfs_pointer_count\':len(lfs),\'pointer_total_bytes\':total,\n                              \'pointer_files\':lfs[:300], \'data_cap_bytes\':DATA_CAP_BYTES}\n    if total > DATA_CAP_BYTES:\n        raise RuntimeError(\'Official required training-data LFS content exceeds %s MB safety cap; see dataset inventory.\'%(DATA_CAP_BYTES//(1024*1024)))\n    if lfs:\n        inc=\',\'.join(\'data/\'+d+\'/**\' for d in sorted(dirs))\n        cmd([\'git\',\'lfs\',\'pull\',\'--include\',inc,\'--exclude\',\'\'],cwd=REPO,timeout=2400)\n    # Confirm that content really replaced LFS pointer (do not use metadata as data).\n    remaining=[]\n    for rel,_ in lfs:\n        with (REPO/rel).open(\'rb\') as f:b=f.read(60)\n        if b.startswith(b\'version https://git-lfs.github.com/spec/v1\'):remaining.append(rel)\n    report[\'data_inventory\'][\'remaining_pointers\']=remaining\n    if remaining:raise RuntimeError(\'Some upstream training-data LFS objects not downloaded\')\n    # Use unmodified official get_train_data and original target metadata.\n    sys.path.insert(0,str(source_dir))\n    old=os.getcwd()\n    try:\n        os.chdir(str(source_dir))\n        data_utils=importlib.import_module(\'data_utils\')\n        data=data_utils.get_train_data(BATCH_SIZE)\n        if len(data)<1:raise RuntimeError(\'Empty official training Sequence\')\n        report[\'data_inventory\'][\'batches\']=int(len(data))\n        report[\'steps\'][\'conditioning\']=\'passed\'\n        return data_utils,data\n    finally:os.chdir(old)\n\ndef generate(tf,generator,data_utils,data):\n    import numpy as np\n    import pandas as pd\n    np.random.seed(SEED)\n    tf.random.set_seed(SEED)\n    sequences=[];all_conditions=[]\n    n_batches=(N_GENERATE+BATCH_SIZE-1)//BATCH_SIZE\n    for batch in range(n_batches):\n        if batch and batch%len(data)==0:data.on_epoch_end()\n        # Original source uses its true amp/non-amp condition batches.\n        _,cond=data[batch%len(data)]\n        cond=np.asarray(cond,dtype=\'float32\')\n        if cond.ndim!=2 or not len(cond):raise RuntimeError(\'Invalid upstream conditioning batch\')\n        lat=np.random.normal(size=(len(cond),256)).astype(\'float32\')\n        sample=generator([tf.convert_to_tensor(lat),tf.convert_to_tensor(cond)])\n        if isinstance(sample,dict):\n            if len(sample)!=1:raise RuntimeError(\'Ambiguous generator outputs\')\n            sample=next(iter(sample.values()))\n        if hasattr(sample,\'numpy\'):sample=sample.numpy()\n        sample=np.asarray(sample)\n        if sample.ndim!=3 or sample.shape[-1]!=23:raise RuntimeError(\'Unexpected generator output \'+str(sample.shape))\n        seqs=data_utils.decode_sequences(sample)\n        sequences.extend(seqs); all_conditions.extend(cond)\n        write_log(\'Batch %s/%s output: %s conditions %s\'%(batch+1,n_batches,sample.shape,cond.shape))\n        if len(sequences)>=N_GENERATE:break\n    # Keep original desired count and nonempty sequences; detect invalid alphabets.\n    sequences=sequences[:N_GENERATE]\n    all_conditions=np.asarray(all_conditions[:N_GENERATE],dtype=\'float32\')\n    alphabet=set(\'ACDEFGHIKLMNPQRSTVWY\')\n    records=[]\n    for i,(seq,cond) in enumerate(zip(sequences,all_conditions)):\n        seq=seq.strip()\n        records.append({\'id\':\'AMPGANv2_%04d\'%(i+1),\'sequence\':seq,\n                        \'length\':len(seq),\'canonical\':bool(seq) and set(seq)<=alphabet,\n                        \'condition_sha256\':hashlib.sha256(cond.tobytes()).hexdigest()})\n    with (OUT/\'ampganv2_generated_candidates_raw.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=[\'id\',\'sequence\',\'length\',\'canonical\',\'condition_sha256\']);w.writeheader();w.writerows(records)\n    with (OUT/\'ampganv2_generated_canonical.fasta\').open(\'w\') as f:\n        for rec in records:\n            if rec[\'canonical\']:f.write(\'>\'+rec[\'id\']+\'\\n\'+rec[\'sequence\']+\'\\n\')\n    report[\'generated_count\']=len(records)\n    report[\'nonempty_count\']=sum(bool(r[\'sequence\']) for r in records)\n    report[\'valid_canonical_count\']=sum(r[\'canonical\'] for r in records)\n    report[\'unique_valid_canonical_count\']=len({r[\'sequence\'] for r in records if r[\'canonical\']})\n    report[\'steps\'][\'generation\']=\'passed\'\n    if report[\'valid_canonical_count\']<1:\n        raise RuntimeError(\'Model ran but yielded no canonical candidate sequences\')\n    write_log(\'Generated %s raw, %s canonical, %s distinct canonical peptides.\' %\n              (len(records),report[\'valid_canonical_count\'],report[\'unique_valid_canonical_count\']))\n\ndef finalize():\n    report[\'timestamp_finished_utc\']=datetime.now(timezone.utc).isoformat()\n    (OUT/\'phase11e2_status.json\').write_text(json.dumps(report,indent=2,default=str))\n    (OUT/\'run.log\').write_text(\'\\n\'.join(LOG)+\'\\n\')\n    (OUT/\'README.txt\').write_text(\'Phase 11E2 independent-source AMPGAN v2 inference.\\n\'\n        \'Outputs are computer-generated candidate sequences, NOT confirmed AMPs.\\n\'\n        \'Official v2.0 checkpoint source and SHA-256 metadata included.\\n\'\n        \'Training data, original checkpoints and original PACE files NOT included.\\n\')\n    with zipfile.ZipFile(ZIP,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for f in sorted(OUT.glob(\'*\')):\n            if f.is_file():z.write(f,f.name)\n    with zipfile.ZipFile(ZIP) as z:\n        bad=z.testzip();assert bad is None,bad\n    write_log(\'OUTPUT: \'+str(ZIP)+\' (\'+str(ZIP.stat().st_size)+\' bytes)\')\n\nif __name__==\'__main__\':\n    try:\n        verify_checkpoint()\n        tf,generator=get_model()\n        du,data=get_source_conditions()\n        generate(tf,generator,du,data)\n        report[\'status\']=\'generated_authentic_amgpan_v2_candidates\'\n    except BaseException as e:\n        report[\'status\']=\'blocked_with_diagnostics\'\n        report[\'error_class\']=type(e).__name__\n        report[\'error_message\']=str(e)\n        (OUT/\'error_traceback.txt\').write_text(traceback.format_exc())\n        write_log(\'BLOCKED: \'+repr(e)+\'\\n\'+traceback.format_exc()[-6000:])\n    finally:\n        finalize()\n'
runner = Path('/content/PACE_Phase11E2_GitLFS_Fixed_Runner.py')
runner.write_text(RUNNER_SOURCE)
print('Runner written:',runner)
exec(compile(RUNNER_SOURCE,str(runner),'exec'),{'__name__':'__main__','__file__':str(runner)})


## 3. Download results for independent verification

Upload `PACE_Phase11E2_Results.zip` even if the generation step reports a new dependency or model compatibility error.


In [ ]:
from pathlib import Path
from google.colab import files
p=Path('/content/PACE_Phase11E2_Results.zip')
assert p.exists(), 'Diagnostic ZIP missing; check runner logs'
print('Results:',p,'bytes:',p.stat().st_size)
files.download(str(p))
